In [2]:
import os
import re
import time
from datetime import datetime
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import pandas as pd
from selenium import webdriver
from selenium.common.exceptions import (
    NoSuchElementException,
    StaleElementReferenceException,
    TimeoutException,
)
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select, WebDriverWait

load_dotenv()

PATH = os.getenv("PORTAL")
BRONZE = os.getenv("BRONZENEW")
BRONZE_LOG = os.getenv("BRONZELOG")

ACTIVITIES = {
    "15": "4(c) Asbestos milling / asbestos-based products",
    "11": "3(b) Cement plants",
    "19": "5(a) Chemical fertilizers",
    "16": "4(d) Chlor-alkali industry",
    "14": "4(b)(ii) Coaltar processing units",
    "13": "4(b) Coke oven plants",
    "25": "5(g) Distilleries",
    "75": "5(ga) Grain based distilleries",
    "26": "5(h) Integrated paint industry",
    "22": "5(d) Manmade fibers manufacturing",
    "10": "3(a) Metallurgical Industries (ferrous and non ferrous)",
    "1": "1(a) Mining of minerals",
    "9": "2(b) Mineral beneficiation",
    "7": "1(e) Nuclear power projects and processing of nuclear fuel",
    "3": "1(b) Off-shore and onshore oil and gas exploration, development and production",
    "79": "2(c) Pellet Plant",
    "20": "5(b) Pesticides industry and pesticide specific intermediates (excluding formulations)",
    "21": "5(c) Petro-chemical complexes (industries based on processing of petroleum fractions",
    "23": "5(e ) Petroleum products and petrochemical based processing such as production of carbon black and electrode grade graphite (processes other than cracking",
    "12": "4(a) Petroleum refining industry",
    "2": "6(a) Pipelines",
    "27": "5(i) Pulp & Paper Industry",
    "30": "7(b) Ship breaking yards including ship breaking units",
    "18": "4(f) Skin/hide processing including the tanning industry",
    "77": "1(a)(ii) Slurry pipelines passing through national parks / sanctuaries / coral reefs, ecologically sensitive areas",
    "17": "4(e) Soda ash Industry",
    "28": "5(j) Sugar Industry",
    "24": "5(f) Synthetic organic chemicals industry",
    "6": "1(d) Thermal Power Plants",
}


def update_silver_log(log_entry: dict) -> None:
    """Logs activity scraping metrics to the Excel file specified by the SILVERLOG env var,

    placing the newest logs at the top.
    """
    if not BRONZE_LOG:
        print("⚠️ SILVERLOG environment variable not set. Skipping log entry.")
        return

    log_file_path = BRONZE_LOG
    if not log_file_path.lower().endswith(".xlsx"):
        log_file_path += ".xlsx"

    log_dir = os.path.dirname(log_file_path)
    if log_dir and not os.path.exists(log_dir):
        os.makedirs(log_dir, exist_ok=True)

    new_df = pd.DataFrame([log_entry])

    columns_order = [
        "Timestamp",
        "State",
        "Activity",
        "Records Fetched",
        "Status",
        "Error Details",
    ]
    new_df = new_df.reindex(columns=columns_order)

    if os.path.exists(log_file_path):
        try:
            existing_df = pd.read_excel(log_file_path, engine="openpyxl")
            combined_df = pd.concat([new_df, existing_df], ignore_index=True)
        except Exception as e:
            print(f"⚠️ Error reading existing log file: {e}. Overwriting file.")
            combined_df = new_df
    else:
        combined_df = new_df

    combined_df.to_excel(log_file_path, engine="openpyxl", index=False)
    print(f"📝 Scraping log updated at: {log_file_path}")


driver = webdriver.Chrome()
driver.get(PATH)
wait = WebDriverWait(driver, 20)

# Click Advance Search
advance_btn = wait.until(
    EC.element_to_be_clickable(
        (By.XPATH, "//button[contains(., 'Show Advance Search')]")
    )
)
driver.execute_script("arguments[0].click();", advance_btn)


# Helper function to trigger Angular events
def trigger_angular_select(element, value=None, by_index=None, by_text=None):
    select_obj = Select(element)

    if value is not None:
        try:
            select_obj.select_by_value(str(value))
        except NoSuchElementException:
            # Fallback to text or index if value matching fails
            if by_text:
                select_obj.select_by_visible_text(by_text)
            elif by_index is not None:
                select_obj.select_by_index(by_index)
    elif by_text is not None:
        select_obj.select_by_visible_text(by_text)
    elif by_index is not None:
        select_obj.select_by_index(by_index)

    driver.execute_script(
        "arguments[0].dispatchEvent(new Event('change', { bubbles: true }));"
        "arguments[0].dispatchEvent(new Event('input', { bubbles: true }));",
        element,
    )


# -----------------------------
# Select Major Clearance Type (Robust Selection)
# -----------------------------
major_clearance_elem = wait.until(
    EC.presence_of_element_located(
        (By.XPATH, "//select[@formcontrolname='majorClearanceType']")
    )
)

# Wait until option elements are populated inside the dropdown
wait.until(lambda d: len(Select(major_clearance_elem).options) > 1)

try:
    trigger_angular_select(
        major_clearance_elem,
        value="1",
        by_index=1,
        by_text="Environment Clearance",
    )
except Exception as e:
    # Fallback to direct index selection
    Select(major_clearance_elem).select_by_index(1)

# Select Issue Authority (MOEFCC)
issue_auth_elem = wait.until(
    EC.presence_of_element_located(
        (By.CSS_SELECTOR, "select[formcontrolname='issueAuthority']")
    )
)
trigger_angular_select(issue_auth_elem, value="MOEFCC", by_text="MOEFCC")

# -----------------------------
# MULTI-SELECT: "Select All" States
# -----------------------------
try:
    # Check if dropdown menu needs to be clicked open first
    select_all_checkbox = driver.find_elements(
        By.XPATH, "//input[@aria-label='multiselect-select-all']"
    )
    if not select_all_checkbox or not select_all_checkbox[0].is_displayed():
        # Click the multi-select container to expand options if collapsed
        dropdown_container = driver.find_element(
            By.XPATH,
            "//*[contains(@class, 'multiselect-dropdown') or contains(@class,"
            " 'dropdown-btn')]",
        )
        driver.execute_script("arguments[0].click();", dropdown_container)
        time.sleep(0.5)

    select_all_input = wait.until(
        EC.presence_of_element_located(
            (By.XPATH, "//input[@aria-label='multiselect-select-all']")
        )
    )

    # Click 'Select All' if not already checked
    if not select_all_input.is_selected():
        driver.execute_script("arguments[0].click();", select_all_input)
        print("🌍 Successfully selected 'Select All' for States.")
        time.sleep(0.5)
except Exception as e:
    print(f"⚠️ Could not click 'Select All' state checkbox: {e}")

all_table_data = []
headers = []

# -----------------------------
# Activity Loop
# -----------------------------
for act_value, act_desc in ACTIVITIES.items():
    print(f"\n⚙️ Searching Activity ID: {act_value} ({act_desc[:30]}...)")

    records_scraped_before = len(all_table_data)
    last_error_reason = ""

    # Retry loop for activity dropdown & year selection
    selection_success = False
    for attempt in range(3):
        try:
            # 1. Select Activity
            activity_dropdown = wait.until(
                EC.presence_of_element_located(
                    (By.XPATH, "//select[@formcontrolname='activityId']")
                )
            )
            trigger_angular_select(activity_dropdown, value=act_value)
            time.sleep(0.3)

            # 2. Select Year = 2026
            year_dropdown = wait.until(
                EC.presence_of_element_located(
                    (By.XPATH, "//select[@formcontrolname='year']")
                )
            )
            wait.until(lambda d: len(Select(year_dropdown).options) > 1)
            trigger_angular_select(
                year_dropdown, value="2026", by_text="2026"
            )
            time.sleep(0.3)

            selection_success = True
            break
        except (StaleElementReferenceException, TimeoutException) as ex:
            last_error_reason = f"Selection attempt {attempt+1} failed: {type(ex).__name__} - {str(ex)}"
            time.sleep(0.5)

    if not selection_success:
        print(
            f"  ❌ Failed to set selection for Activity {act_value} / Year"
            " 2026. Skipping..."
        )
        update_silver_log({
            "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "State": "ALL STATES",
            "Activity": act_desc,
            "Records Fetched": 0,
            "Status": "Failed",
            "Error Details": f"Dropdown selection failed after 3 attempts. Last error: {last_error_reason}",
        })
        continue

    search_button = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//button[@type='submit' and contains(.,'Search')]")
        )
    )

    existing_tables = driver.find_elements(By.ID, "excel-table")
    old_table = existing_tables[0] if existing_tables else None

    driver.execute_script("arguments[0].click();", search_button)

    if old_table:
        try:
            wait.until(EC.staleness_of(old_table))
        except Exception:
            time.sleep(1)

    try:
        wait.until(EC.visibility_of_element_located((By.ID, "excel-table")))
        time.sleep(0.5)
    except TimeoutException:
        print(
            f"  ℹ️ No results found for Activity: {act_value} (Year: 2026)."
            " Proceeding..."
        )
        update_silver_log({
            "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "State": "ALL STATES",
            "Activity": act_desc,
            "Records Fetched": 0,
            "Status": "No Records",
            "Error Details": "Table 'excel-table' not visible within timeout (likely 0 results).",
        })
        continue

    # -----------------------------
    # PAGINATION
    # -----------------------------
    page_num = 1
    pagination_error = None

    while True:
        rows_elements = driver.find_elements(
            By.XPATH, "//table[@id='excel-table']/tbody/tr"
        )

        if not rows_elements:
            break

        first_row_text = rows_elements[0].text.lower()
        if "no record" in first_row_text or "no data" in first_row_text:
            break

        if not headers:
            html = driver.page_source
            soup = BeautifulSoup(html, "html.parser")
            table = soup.find("table", {"id": "excel-table"})
            if table and table.find("thead"):
                for th in table.find("thead").find_all("th"):
                    headers.append(th.get_text(strip=True))

        total_rows = len(rows_elements)
        print(f"  📊 Page {page_num}: Scraping {total_rows} records...")

        for row in rows_elements:
            try:
                cols = row.find_elements(By.TAG_NAME, "td")
                row_data = [col.text.strip() for col in cols]

                if not row_data or len(row_data) <= 1:
                    continue

                row_data.append("ALL STATES")
                row_data.append(act_desc)
                all_table_data.append(row_data)
            except Exception:
                continue

        # Pagination control
        try:
            next_buttons = driver.find_elements(
                By.XPATH, "//button[@aria-label='Next page']"
            )
            if not next_buttons:
                break

            next_btn = next_buttons[0]

            is_disabled = (
                next_btn.get_attribute("disabled") in ["true", "disabled", True]
                or next_btn.get_attribute("aria-disabled") == "true"
                or "mat-button-disabled"
                in (next_btn.get_attribute("class") or "")
            )

            if is_disabled:
                print(
                    f"  🎉 Reached final page ({page_num}) for Activity"
                    f" {act_value}."
                )
                break

            current_signature = (
                rows_elements[0].text if rows_elements else ""
            )

            driver.execute_script("arguments[0].click();", next_btn)
            page_num += 1

            def wait_for_page_transition(d):
                try:
                    new_rows = d.find_elements(
                        By.XPATH, "//table[@id='excel-table']/tbody/tr"
                    )
                    if not new_rows:
                        return False
                    return new_rows[0].text != current_signature
                except (StaleElementReferenceException, NoSuchElementException):
                    return False

            WebDriverWait(driver, 20).until(wait_for_page_transition)
            time.sleep(0.5)

        except TimeoutException:
            pagination_error = f"Timeout waiting for page {page_num} transition."
            print(
                f"  ⚠️ Timeout on page {page_num}. Moving to next activity..."
            )
            break
        except Exception as ex:
            pagination_error = f"Exception on page {page_num}: {type(ex).__name__} - {str(ex)}"
            print(f"  ⚠️ Pagination exception on page {page_num}: {ex}")
            break

    activity_records_count = len(all_table_data) - records_scraped_before
    status = "Success" if not pagination_error else "Partial Success / Interrupted"

    update_silver_log({
        "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "State": "ALL STATES",
        "Activity": act_desc,
        "Records Fetched": activity_records_count,
        "Status": status,
        "Error Details": pagination_error if pagination_error else "None",
    })

    time.sleep(0.3)


# -----------------------------
# Save Results
# -----------------------------
if all_table_data:
    expected_header_count = len(all_table_data[0])

    if len(headers) < expected_header_count:
        headers.append("State_Name")
    if len(headers) < expected_header_count:
        headers.append("Activity Description")

    df = pd.DataFrame(all_table_data, columns=headers[:expected_header_count])

    # Remove illegal Excel control characters (\x00-\x08, \x0B-\x0C, \x0E-\x1F)
    illegal_xml_chars_re = re.compile(r"[\x00-\x08\x0B-\x0C\x0E-\x1F]")
    df = df.map(
        lambda x: illegal_xml_chars_re.sub("", x) if isinstance(x, str) else x
    )

    print("\n--- Final Extracted Dataset Preview ---")
    print(df.head())

    # Build folder path using SILVERNEW env var and current date (YYYY-MM-DD)
    if not BRONZE:
        print("⚠️ SILVERNEW environment variable not set. Defaulting to local directory.")
        base_dir = "."
    else:
        base_dir = BRONZE

    date_folder = datetime.now().strftime("%Y-%m-%d")
    output_dir = os.path.join(base_dir, date_folder)

    if not os.path.exists(output_dir):
        os.makedirs(output_dir, exist_ok=True)

    # Generate timestamped filename
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    timestamped_filename = f"MOEFCC_{timestamp}.xlsx"
    file_path = os.path.join(output_dir, timestamped_filename)

    df.to_excel(file_path, engine="openpyxl", index=False)
    print(f"\n✅ Total {len(df)} rows scraped and saved to {file_path}")
else:
    print(
        "\n❌ Automation complete. Zero data entries found across the"
        " state/activity matrix."
    )

driver.quit()

🌍 Successfully selected 'Select All' for States.

⚙️ Searching Activity ID: 15 (4(c) Asbestos milling / asbest...)
  📊 Page 1: Scraping 1 records...
  🎉 Reached final page (1) for Activity 15.
📝 Scraping log updated at: F:\Chimney Work\Marketing\LeadGen\Data Architecture\1 - Bronze\Logs.xlsx

⚙️ Searching Activity ID: 11 (3(b) Cement plants...)
  📊 Page 1: Scraping 10 records...
  📊 Page 2: Scraping 10 records...
  📊 Page 3: Scraping 10 records...
  📊 Page 4: Scraping 10 records...
  📊 Page 5: Scraping 1 records...
  🎉 Reached final page (5) for Activity 11.
📝 Scraping log updated at: F:\Chimney Work\Marketing\LeadGen\Data Architecture\1 - Bronze\Logs.xlsx

⚙️ Searching Activity ID: 19 (5(a) Chemical fertilizers...)
  📊 Page 1: Scraping 10 records...
  📊 Page 2: Scraping 10 records...
  📊 Page 3: Scraping 10 records...
  🎉 Reached final page (3) for Activity 19.
📝 Scraping log updated at: F:\Chimney Work\Marketing\LeadGen\Data Architecture\1 - Bronze\Logs.xlsx

⚙️ Searching Activity ID